In [17]:
import numpy as np
import scipy.linalg
from scipy.sparse.linalg import lsmr
import time

 

np.random.seed(42)
n = 100  # Number of samples
d = 100     # Number of features

X = np.random.randn(n, d)
true_beta = np.random.randn(d)
y = X @ true_beta + np.random.randn(n)  # Generating response with noise

M = X.T @ X  # Positive definite matrix
b = X.T @ y
print(M.shape, b.shape)

results = {}
print("1")
# Option 1: NumPy lstsq
start = time.time()
x_np = np.linalg.lstsq(M, b, rcond=None)[0]
results['NumPy lstsq'] = time.time() - start
print("2")
# Option 2: SciPy lstsq
start = time.time()
x_scipy = scipy.linalg.lstsq(M, b, cond=None)[0]
results['SciPy lstsq'] = time.time() - start
print("3")
# Option 3: Normal Equations (if M is full-rank)
start = time.time()
M_T_M = M.T @ M
M_T_b = M.T @ b
x_ne = np.linalg.solve(M_T_M, M_T_b)
results['Normal Equations'] = time.time() - start
print("4")
# Option 4: Iterative solver (lsmr)
start = time.time()
x_lsmr = lsmr(M, b)[0]
results['SciPy lsmr'] = time.time() - start

 
# Display results
for method, duration in results.items():
    print(f"{method}: {duration:.4f} seconds")


(100, 100) (100,)
1
2
3
4
NumPy lstsq: 0.1325 seconds
SciPy lstsq: 0.2994 seconds
Normal Equations: 0.0003 seconds
SciPy lsmr: 0.0027 seconds


In [43]:
import numpy as np
import scipy.linalg
from scipy.sparse.linalg import lsmr
import time

def compute_group_means_v1(data, Z, folds, fold_val, K):
    """
    Compute group means for data (which can be X or Y) for a given fold.
    data: array of shape (n_total, d) or (n_total,) for Y.
    Z: group labels (n_total,)
    folds: fold indicator array (n_total,)
    fold_val: the fold for which to compute the means (0 or 1)
    K: number of groups
    """
    idx = (folds == fold_val)
    Z_idx = Z[idx]  # Extract group labels for this fold only
    if data.ndim == 2:
        d = data.shape[1]
        group_sum = np.zeros((K, d))
        counts = np.zeros(K, dtype=int)

        # Fast bincount for each column (avoiding np.add.at)
        for j in range(d):
            group_sum[:, j] = np.bincount(Z_idx, weights=data[idx, j], minlength=K)
        
        counts = np.bincount(Z_idx, minlength=K)

    else:  # data is 1D
        group_sum = np.bincount(Z_idx, weights=data[idx], minlength=K)
        counts = np.bincount(Z_idx, minlength=K)

    counts[counts == 0] = 1  # Avoid division by zero
    group_means = group_sum / counts[:, None] if data.ndim == 2 else group_sum / counts
    
    return group_means

 

def compute_group_means_v2(data, Z, folds, fold_val, K):
    '''Optimized version of compute_group_means'''
    idx = (folds == fold_val)
    Z_idx = Z[idx]

    counts = np.bincount(Z_idx, minlength=K)
    counts[counts == 0] = 1  # Avoid division by zero

    if data.ndim == 2:
        group_sum = np.vstack([
            np.bincount(Z_idx, weights=data[idx, j], minlength=K)
            for j in range(data.shape[1])
        ]).T
        group_means = group_sum / counts[:, None]
    else:
        group_sum = np.bincount(Z_idx, weights=data[idx], minlength=K)
        group_means = group_sum / counts

    return group_means




def compute_group_means_v3(data, Z, folds, fold_val, K):
    mask = (folds == fold_val)
    Z_sub = Z[mask]
    data_sub = data[mask]

    if data_sub.ndim == 1:
        df = pd.DataFrame({'Z': Z_sub, 'val': data_sub})
        means_series = df.groupby('Z')['val'].mean()
        # Reindex to get K rows
        means_array = np.zeros(K, dtype=data_sub.dtype)
        means_array[means_series.index] = means_series.values
        return means_array
    else:
        df_dict = {'Z': Z_sub}
        for j in range(data_sub.shape[1]):
            df_dict[f'col{j}'] = data_sub[:, j]
        df = pd.DataFrame(df_dict)
        means_df = df.groupby('Z').mean()
        # Reindex to get K rows
        means_array = np.zeros((K, data_sub.shape[1]), dtype=data_sub.dtype)
        for j in range(data_sub.shape[1]):
            colmean = f'col{j}'
            if colmean in means_df:
                means_array[means_df.index, j] = means_df[colmean].values
        return means_array


# Testing both functions
np.random.seed(0)
n = 50
K = 2000
n_total = K * n 
d = 65

Z = np.random.randint(0, K, size=n_total)
folds = np.random.randint(0, 2, size=n_total)
data = np.random.randn(n_total, d)

start_v1 = time.time()
group_means_v1 = compute_group_means_v1(data, Z, folds, fold_val=0, K=K)
time_v1 = time.time() - start_v1

start_v2 = time.time()
group_means_v2 = compute_group_means_v2(data, Z, folds, fold_val=0, K=K)
time_v2 = time.time() - start_v2


start_v3 = time.time()
group_means_v3 = compute_group_means_v3(data, Z, folds, fold_val=0, K=K)
time_v3 = time.time() - start_v3


print(f"Original Version Time: {time_v1:.4f} seconds")
print(f"Optimized Version Time: {time_v2:.4f} seconds")
print(f"Optimized Version Time: {time_v3:.4f} seconds")

# Checking correctness
assert np.allclose(group_means_v1, group_means_v2), "The optimized version produces different results!"


Original Version Time: 0.0652 seconds
Optimized Version Time: 0.0974 seconds
Optimized Version Time: 0.0373 seconds


In [31]:
import pandas as pd
import numpy as np

def compute_group_means_pandas(data, Z, folds, fold_val, K):
    mask = (folds == fold_val)
    Z_sub = Z[mask]
    data_sub = data[mask]

    if data_sub.ndim == 1:
        df = pd.DataFrame({'Z': Z_sub, 'val': data_sub})
        means_series = df.groupby('Z')['val'].mean()
        # Reindex to get K rows
        means_array = np.zeros(K, dtype=data_sub.dtype)
        means_array[means_series.index] = means_series.values
        return means_array
    else:
        df_dict = {'Z': Z_sub}
        for j in range(data_sub.shape[1]):
            df_dict[f'col{j}'] = data_sub[:, j]
        df = pd.DataFrame(df_dict)
        means_df = df.groupby('Z').mean()
        # Reindex to get K rows
        means_array = np.zeros((K, data_sub.shape[1]), dtype=data_sub.dtype)
        for j in range(data_sub.shape[1]):
            colmean = f'col{j}'
            if colmean in means_df:
                means_array[means_df.index, j] = means_df[colmean].values
        return means_array


In [27]:
5000000/K

500.0